In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
import os

In [ ]:
# PATH
csv_path = os.path.join('..', 'data', 'emotify_final_dataset.csv')
base_folder = os.path.join('..', 'data','Emotify_npy') 


def load_dataset(csv_path, base_folder='data'):
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"CSV file not found: {csv_path}")
        
    df = pd.read_csv(csv_path)
    X = []
    emotion_cols = ['amazement', 'solemnity', 'tenderness', 'nostalgia', 'calmness', 
                    'power', 'joyful_activation', 'tension', 'sadness']
    y = df[emotion_cols].values

    print(f"loading {len(df)} .npy files...")
    for path in df['npy_path']:
        full_path = os.path.join(base_folder, path)
        data = np.load(full_path)
        
        # Normalization of the data to the range [0, 1]
        data_min, data_max = data.min(), data.max()
        if data_max - data_min != 0:
            data = (data - data_min) / (data_max - data_min)
        X.append(data)
    
    X = np.array(X)
    X = np.expand_dims(X, axis=-1) 
    return X, y, emotion_cols

arxiko cnn

In [ ]:

def build_cnn_model(input_shape=(128, 1292, 1)):
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(9, activation='sigmoid') # 9 feels
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [ ]:



# Α. LOAD
X, y, labels = load_dataset(csv_path, base_folder)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Β. Model Creation
model = build_cnn_model(input_shape=X_train.shape[1:])
model.summary()

# Γ. Training
print("\nTraining the model...")
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=16,
    validation_data=(X_test, y_test),
    verbose=1
)

# Δ. SAVING MODEL
model.save('music_emotion_cnn.h5')
print("\nSuccess! The model has been saved.")

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Test')
plt.title('Accuracy'); plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Test')
plt.title('Loss'); plt.legend()
plt.show()